# GAN

GAN は生成器 `G` と識別器 `D` を競わせ、データ分布を直接の尤度なしで近づける方法である。`D` は本物と偽物を見分け、`G` は `D` が本物と判定しやすいサンプルを作る。

損失名だけではなく、対戦の力学が学習の安定性を左右する。識別器が強すぎると生成器の勾配が弱くなり、生成器の表現力が足りないと mode collapse のように一部の山だけを出す。

In [ ]:
import math
import random
import statistics

random.seed(21)


def sample_real(n):
    out = []
    for _ in range(n):
        if random.random() < 0.5:
            out.append(random.gauss(-2.0, 0.35))
        else:
            out.append(random.gauss(1.8, 0.42))
    return out


def sample_z(n):
    return [random.gauss(0.0, 1.0) for _ in range(n)]

real_ref = sample_real(1800)


def summarize(xs):
    return {
        'mean': round(statistics.mean(xs), 3),
        'std': round(statistics.pstdev(xs), 3),
        'left': round(sum(x < 0 for x in xs) / len(xs), 3),
    }

print('real:', summarize(real_ref))

1 次元データでも、二峰性を使うと collapse が見えやすい。線形生成器 `G(z)=a z+b` は必ず単峰に近く、左右 2 つの山を同時に表すには器が足りない。生成器の形が目的分布より単純すぎると、損失を工夫しても表せない構造が残る。

In [ ]:
def generator_linear(z, theta):
    a, b = theta
    return a * z + b


def generator_piecewise(z, theta):
    a_l, b_l, a_r, b_r = theta
    if z < 0:
        return a_l * z + b_l
    return a_r * z + b_r


def discriminator(x, phi):
    w, b = phi
    return w * x + b


def sigmoid(x):
    if x >= 0:
        e = math.exp(-x)
        return 1.0 / (1.0 + e)
    e = math.exp(x)
    return e / (1.0 + e)

lin0 = [0.2, 0.0]
pw0 = [0.4, -1.0, 0.4, 1.0]
print('linear fake:', summarize([generator_linear(z, lin0) for z in sample_z(1000)]))
print('piecewise fake:', summarize([generator_piecewise(z, pw0) for z in sample_z(1000)]))

識別器の目的は、本物の `log D(x)` と偽物の `log(1-D(G(z)))` を大きくすること。生成器には `log D(G(z))` を大きくする non-saturating 目的を使う。

In [ ]:
def log_prob(p, eps=1e-8):
    return math.log(max(eps, min(1.0 - eps, p)))


def gan_scores(theta, phi, real_batch, z_batch, gen_fn):
    fake = [gen_fn(z, theta) for z in z_batch]
    d_real = [sigmoid(discriminator(x, phi)) for x in real_batch]
    d_fake = [sigmoid(discriminator(x, phi)) for x in fake]
    d_obj = sum(log_prob(p) for p in d_real) / len(d_real)
    d_obj += sum(log_prob(1.0 - p) for p in d_fake) / len(d_fake)
    g_obj = sum(log_prob(p) for p in d_fake) / len(d_fake)
    return d_obj, g_obj, fake

real_batch = sample_real(256)
z_batch = sample_z(256)
print('linear objectives:', [round(v, 3) for v in gan_scores(lin0, [0.4, 0.0], real_batch, z_batch, generator_linear)[:2]])

有限差分で勾配を作り、`D` と `G` を交互に更新する。深いモデルでは自動微分を使うが、小さな実装では更新方向の意味が見える。`D` は本物と偽物の分離を強め、`G` は偽物が本物側へ寄るように動くため、片方だけを最適化しても目的分布には近づかない。

In [ ]:
def finite_grad(fn, params, h=1e-4):
    out = []
    for i in range(len(params)):
        plus = params[:]
        minus = params[:]
        plus[i] += h
        minus[i] -= h
        out.append((fn(plus) - fn(minus)) / (2.0 * h))
    return out


def w1_distance(a, b):
    aa = sorted(a)
    bb = sorted(b)
    n = min(len(aa), len(bb))
    return sum(abs(aa[i] - bb[i]) for i in range(n)) / n


def train_gan(theta, phi, gen_fn, steps=150, lr_d=0.08, lr_g=0.05):
    theta = theta[:]
    phi = phi[:]
    history = []
    for step in range(steps):
        real_batch = sample_real(128)
        z_batch = sample_z(128)

        def d_fn(p):
            return gan_scores(theta, p, real_batch, z_batch, gen_fn)[0]

        d_grad = finite_grad(d_fn, phi)
        for i in range(len(phi)):
            phi[i] += lr_d * d_grad[i]

        z_batch = sample_z(128)
        real_batch = sample_real(128)

        def g_fn(t):
            return gan_scores(t, phi, real_batch, z_batch, gen_fn)[1]

        g_grad = finite_grad(g_fn, theta)
        for i in range(len(theta)):
            theta[i] += lr_g * g_grad[i]

        if step % 30 == 0 or step == steps - 1:
            fake = [gen_fn(z, theta) for z in sample_z(1200)]
            history.append((step, theta[:], phi[:], w1_distance(real_ref[:len(fake)], fake), summarize(fake)))
    return theta, phi, history

lin_theta, lin_phi, lin_history = train_gan(lin0, [0.4, 0.0], generator_linear)
for row in lin_history:
    step, theta, phi, w1, stat = row
    print(step, [round(v, 3) for v in theta], 'w1=', round(w1, 3), stat)

線形生成器は左右の山を同時に作れない。分岐生成器なら `z<0` と `z>=0` で別の写像を持てるため、二峰性を表しやすくなる。

In [ ]:
pw_theta, pw_phi, pw_history = train_gan(pw0, [0.4, 0.0], generator_piecewise, steps=170, lr_g=0.045)
for row in pw_history:
    step, theta, phi, w1, stat = row
    print(step, [round(v, 3) for v in theta], 'w1=', round(w1, 3), stat)

lin_fake = [generator_linear(z, lin_theta) for z in sample_z(1800)]
pw_fake = [generator_piecewise(z, pw_theta) for z in sample_z(1800)]
print('linear final:', summarize(lin_fake), 'w1=', round(w1_distance(real_ref, lin_fake), 3))
print('piecewise final:', summarize(pw_fake), 'w1=', round(w1_distance(real_ref, pw_fake), 3))

minimax 目的では `D(fake)` が小さい初期段階で生成器の勾配が弱くなりやすい。non-saturating 目的は、偽物が見破られているほど大きな修正信号を返す。損失の違いは名前の違いではなく、生成器へ返る勾配の大きさの違いとして現れる。

In [ ]:
def grad_minimax_logit(p):
    return -p


def grad_nonsat_logit(p):
    return -(1.0 - p)

for p in [0.001, 0.01, 0.05, 0.1, 0.5, 0.9]:
    print('D(fake)=', p, 'minimax |grad|=', round(abs(grad_minimax_logit(p)), 3), 'non-sat |grad|=', round(abs(grad_nonsat_logit(p)), 3))

LSGAN は二乗誤差で識別器を学習し、WGAN は確率ではなく critic の平均差を見る。どちらも、通常 GAN の勾配や距離の扱いづらさを減らすための設計である。

In [ ]:
def lsgan_values(theta, phi, real_batch, z_batch, gen_fn):
    fake = [gen_fn(z, theta) for z in z_batch]
    d_real = [discriminator(x, phi) for x in real_batch]
    d_fake = [discriminator(x, phi) for x in fake]
    d_loss = sum((v - 1.0) ** 2 for v in d_real) / len(d_real)
    d_loss += sum(v ** 2 for v in d_fake) / len(d_fake)
    g_loss = sum((v - 1.0) ** 2 for v in d_fake) / len(d_fake)
    return d_loss, g_loss


def wgan_values(theta, phi, real_batch, z_batch, gen_fn):
    fake = [gen_fn(z, theta) for z in z_batch]
    c_real = [discriminator(x, phi) for x in real_batch]
    c_fake = [discriminator(x, phi) for x in fake]
    critic_gap = sum(c_real) / len(c_real) - sum(c_fake) / len(c_fake)
    return critic_gap, -sum(c_fake) / len(c_fake)

real_batch = sample_real(256)
z_batch = sample_z(256)
print('LSGAN linear:', [round(v, 3) for v in lsgan_values(lin_theta, lin_phi, real_batch, z_batch, generator_linear)])
print('WGAN linear:', [round(v, 3) for v in wgan_values(lin_theta, lin_phi, real_batch, z_batch, generator_linear)])
print('LSGAN piecewise:', [round(v, 3) for v in lsgan_values(pw_theta, pw_phi, real_batch, z_batch, generator_piecewise)])
print('WGAN piecewise:', [round(v, 3) for v in wgan_values(pw_theta, pw_phi, real_batch, z_batch, generator_piecewise)])

GAN を読むときは、生成器が目的分布を表せるか、識別器が勝ちすぎていないか、生成器へ十分な勾配が返っているかを分けて見る。損失の変更は、どの失敗を減らすための変更なのかを確認する。